# Pipeline de Análise de Churn — Spotify Dataset

**Disciplina:** Data Integration e Pipelines  
**Arquitetura:** Medallion (Bronze → Silver → Gold)  
**Ambiente:** Databricks Free Edition (Unity Catalog + Serverless)

---

## Contexto de negócio

Vocês foram contratados como engenheiros de dados de uma plataforma de streaming de música.  
A empresa precisa de um **pipeline automatizado** que identifique usuários com alto risco de cancelar a assinatura (churn).

O pipeline deverá:

1. **Ingerir** os dados brutos de comportamento dos usuários (camada **Bronze**)  
2. **Limpar e enriquecer** com indicadores de engajamento (camada **Silver**)  
3. **Gerar métricas** e identificar perfis de churn (camada **Gold**)  
4. **Validar** a qualidade dos dados a cada etapa  
5. **Registrar alertas** quando anomalias forem detectadas  

### Arquitetura do pipeline


spotify_churn_dataset.csv + events.csv  (Volume do Unity Catalog)

```
+-----------+     +-----------+     +-----------+     +--------+
|  BRONZE   |---->|  SILVER   |---->|   GOLD    |---->| ALERTA |
| (raw data)|     | (limpo +  |     |(metricas +|     |(tabela |
| tabela UC |     | enriquec) |     | segmentos)|     |  +log) |
+-----------+     +-----------+     +-----------+     +--------+
```

Todas as tabelas são **gerenciadas pelo Unity Catalog** - padrão da Free Edition.
---

### Dataset: Spotify Dataset for Churn Analysis (Kaggle)

**Fonte:** [Kaggle - Spotify Dataset for Churn Analysis](https://www.kaggle.com/datasets/nabihazahid/spotify-dataset-for-churn-analysis)

#### Arquivo 1: `spotify_churn_dataset.csv` - Perfil dos usuários

| Coluna                  | Tipo    | Descrição                                         |
|-------------------------|---------|---------------------------------------------------|
| `user_id`               | int     | Identificador único do usuário                    |
| `gender`                | string  | Gênero (Male / Female / Other)                    |
| `age`                   | int     | Idade do usuário                                  |
| `country`               | string  | País do usuário (código ISO)                      |
| `subscription_type`     | string  | Tipo de assinatura (Free / Premium / Family)      |
| `listening_time`        | int     | Minutos totais de escuta no período               |
| `songs_played_per_day`  | int     | Média de músicas reproduzidas por dia             |
| `skip_rate`             | double  | Taxa de músicas puladas (0.0 a 1.0)               |
| `device_type`           | string  | Dispositivo principal (Mobile / Desktop / Web)    |
| `ads_listened_per_week` | int     | Anúncios ouvidos por semana (usuários Free)       |
| `offline_listening`     | int     | Usa modo offline? (0 = não, 1 = sim)              |
| `is_churned`            | int     | Cancelou? (0 = ativo, 1 = churned) - campo-alvo   |

#### Arquivo 2: `events.csv` - Histórico de eventos de comportamento

| Coluna                | Tipo    | Descrição                                         |
|-----------------------|---------|---------------------------------------------------|
| `user_id`             | int     | Identificador do usuário                          |
| `event_type`          | string  | Tipo de evento (play, skip, like, share, etc.)    |
| `event_timestamp`     | string  | Data e hora do evento                             |
| `song_id`             | string  | ID da música envolvida                            |
| `device_type`         | string  | Dispositivo usado                                 |
| `session_duration_min`| double  | Duração da sessão em minutos                      |

---

### Justificativa de negócio

Churn é um dos principais KPIs de qualquer serviço de assinatura.  
O custo de adquirir um novo cliente é 5x maior do que reter um existente.  
Um pipeline bem estruturado permite que o time de CRM identifique usuários em risco **antes** do cancelamento e ative campanhas de retenção.

**Frequência de atualização:** Diária (comportamento muda a cada sessão)  
**Consumidores finais:** Times de CRM, Marketing e Produto

### Pré-requisitos
- Arquivos `spotify_churn_dataset.csv` e `events.csv` carregados no Volume `spotify_churn`
- Estrutura: `/Volumes/workspace/default/spotify_churn/`

**Tempo estimado:** 60 minutos

---
## PARTE 1 - Configuracao Inicial

In [0]:
%sh
# Cria a pasta no DBFS (se não existir)
mkdir -p /Volumes/workspace/study/spotify/

# Faz o download do ZIP direto do GitHub
wget -O /Volumes/workspace/study/spotify/datasets.zip \
  https://raw.githubusercontent.com/roderjan/Lab_Pipeline_Spotify/main/datasets/datasets.zip

mkdir: cannot create directory ‘/Volumes/workspace/study/spotify/’: Permission denied
/Volumes/workspace/study/spotify/datasets.zip: Permission denied


In [0]:
%sh
unzip -o /Volumes/workspace/study/spotify/datasets.zip -d /Volumes/workspace/study/spotify/

unzip:  cannot find or open /Volumes/workspace/study/spotify/datasets.zip, /Volumes/workspace/study/spotify/datasets.zip.zip or /Volumes/workspace/study/spotify/datasets.zip.ZIP.


In [0]:
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    max as spark_max, min as spark_min, round as spark_round,
    current_timestamp, percentile_approx, stddev, countDistinct,
    coalesce
)
from pyspark.sql.types import DoubleType, IntegerType, StringType
from datetime import datetime
import json as json_lib

print("=" * 60)
print("  VERIFICACAO DO AMBIENTE")
print("=" * 60)
print(f"  Spark version:    {spark.version}")
print(f"  Data/hora:        {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Ambiente:         Databricks Free Edition")
print(f"  Compute:          Serverless (automatico)")
print(f"  Storage:          Unity Catalog (tabelas gerenciadas)")
print("=" * 60)
print("  OK - Ambiente pronto!")

  VERIFICACAO DO AMBIENTE
  Spark version:    4.1.0
  Data/hora:        2026-03-24 01:35:22
  Ambiente:         Databricks Free Edition
  Compute:          Serverless (automatico)
  Storage:          Unity Catalog (tabelas gerenciadas)
  OK - Ambiente pronto!


---
## PARTE 2 - Leitura dos Datasets

Os arquivos foram carregados no Volume `spotify_churn` do Unity Catalog.
Vamos ler ambos e fazer uma exploracao inicial antes de qualquer transformacao.

In [0]:
# ============================================================
# Caminhos dos datasets no Volume do Unity Catalog
# Estrutura: /Volumes/{catalogo}/{esquema}/{volume}/{arquivo}
# ============================================================

# ATENCAO: Permissao negada para leitura do Volume 'workspace.study.spotify'
# Solucao alternativa: Carregar arquivos via upload manual ou outro caminho acessivel

PATH_USERS  = "/dbfs/FileStore/spotify_churn_dataset.csv"
PATH_EVENTS = "/dbfs/FileStore/events.csv"

# Leitura dos arquivos
df_users_raw  = spark.read.csv(PATH_USERS,  header=True, inferSchema=True)
df_events_raw = spark.read.csv(PATH_EVENTS, header=True, inferSchema=True)

print(f"Datasets carregados!")
print()
print(f"Usuarios (spotify_churn_dataset.csv):")
print(f"   Registros: {df_users_raw.count():,}")
print(f"   Colunas:   {df_users_raw.columns}")
print()
print(f"Eventos (events.csv):")
print(f"   Registros: {df_events_raw.count():,

Datasets carregados!

Usuarios (spotify_churn_dataset.csv):


---------------------------------------------------------------------------
SparkConnectGrpcException                 Traceback (most recent call last)
File <command-7104142458445078>, line 17
     15 print()
     16 print(f"Usuarios (spotify_churn_dataset.csv):")
---> 17 print(f"   Registros: {df_users_raw.count():,}")
     18 print(f"   Colunas:   {df_users_raw.columns}")
     19 print()

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/dataframe.py:300, in DataFrame.count(self)
    299 def count(self) -> int:
--> 300     table, _ = self.agg(F._invoke_function("count", F.lit(1)))._to_table()
    301     return table[0][0].as_py()

File /databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/dataframe.py:1971, in DataFrame._to_table(self)
   1969 def _to_table(self) -> Tuple["pa.Table", Optional[StructType]]:
   1970     query = self._plan.to_proto(self._session.client)
-> 1971     table, schema, self._execution_info = self._session.client.to_table(

---
## PARTE 3 - Camada BRONZE (Dados Brutos)

A Bronze armazena os dados **exatamente como vieram da fonte**,
sem nenhuma transformacao. Apenas adicionamos metadados de controle:
quando foi ingerido, de onde veio, qual versao do pipeline.

### Por que guardar os dados brutos?

No contexto de churn, dados historicos sao essenciais: padroes de
comportamento de meses atras podem ser mais preditivos que os recentes.
Se transformarmos sem guardar o original, perdemos essa capacidade de
reprocessamento. A Bronze e o "cofre" - sempre e possivel recalcular
tudo a partir dela.

**Conceito ELT:** carregamos primeiro (Load), transformamos depois.
Isso e o oposto do ETL legado - e a pratica moderna em Data Lakes.

In [0]:
# ============================================================
# BRONZE: Usuarios - salvar com metadados de controle
# ============================================================

# Verifica se a tabela existe
tabela_bronze = "workspace.study.bronze_usuarios_spotify_rev2"
tabelas_existentes = [t.name for t in spark.catalog.listTables("study")]

if tabela_bronze.split(".")[-1] not in tabelas_existentes:
    spark.sql(f"""
        CREATE TABLE {tabela_bronze} (
            user_id INT,
            gender STRING,
            age INT,
            country STRING,
            subscription_type STRING,
            listening_time INT,
            songs_played_per_day INT,
            skip_rate DOUBLE,
            device_type STRING,
            ads_listened_per_week INT,
            offline_listening INT,
            is_churned INT,
            _ingestao_timestamp TIMESTAMP,
            _fonte STRING,
            _pipeline_version STRING
        )
        USING DELTA
    """)

df_bronze_users = df_users_raw \
    .withColumn("_ingestao_timestamp", current_timestamp()) \
    .withColumn("_fonte", lit("kaggle/spotify_churn_dataset.csv")) \
    .withColumn("_pipeline_version", lit("1.0.0"))

df_bronze_users.write \
    .mode("overwrite") \
    .saveAsTable(tabela_bronze)

total_bronze_users = spark.table(tabela_bronze).count()
print(f"Bronze Usuarios criada!")
print(f"   Registros:  {total_bronze_users:,}")
print(f"   Formato:    Delta Lake (gerenciada pelo Unity Catalog)")
print(f"   Tabela:     {tabela_bronze}")

In [0]:
# ============================================================
# BRONZE: Eventos - salvar com metadados de controle
# ============================================================

df_bronze_events = df_events_raw \
    .withColumn("_ingestao_timestamp", current_timestamp()) \
    .withColumn("_fonte", lit("kaggle/events.csv")) \
    .withColumn("_pipeline_version", lit("1.0.0"))

df_bronze_events.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_eventos_spotify")

total_bronze_events = spark.table("workspace.default.bronze_eventos_spotify").count()
print(f"Bronze Eventos criada!")
print(f"   Registros:  {total_bronze_events:,}")
print(f"   Formato:    Delta Lake (gerenciada pelo Unity Catalog)")
print(f"   Tabela:     workspace.default.bronze_eventos_spotify")

### Data Profiling - conhecendo os dados antes de transformar

In [0]:
# Visualizar amostra dos usuarios
print("Amostra - Usuarios:")
display(spark.table("workspace.default.bronze_usuarios_spotify").limit(10))

In [0]:
# Estatisticas descritivas dos usuarios
print("Estatisticas descritivas - Usuarios:")
display(spark.table("workspace.default.bronze_usuarios_spotify").describe())

In [0]:
# Relatorio de completude - Usuarios
df_b  = spark.table("workspace.default.bronze_usuarios_spotify")
total = df_b.count()

print("=" * 70)
print(f"  RELATORIO DE PROFILING - BRONZE USUARIOS")
print(f"     Total de registros: {total:,}")
print("=" * 70)

for c in df_b.columns:
    if c.startswith("_"):
        continue
    nulos = df_b.filter(col(c).isNull()).count()
    pct   = (nulos / total) * 100
    icon  = "ATENCAO" if pct > 5 else "AVISO" if pct > 0 else "OK"
    print(f"  [{icon}] {c:40s} | Nulos: {nulos:>8,} ({pct:5.1f}%)")

print("=" * 70)

In [0]:
# Distribuicao da variavel-alvo: is_churned
print("DISTRIBUICAO DE CHURN NO DATASET:")
display(
    df_b.groupBy("is_churned")
    .agg(
        count("*").alias("quantidade"),
        spark_round(count("*") / total * 100, 2).alias("percentual")
    )
    .orderBy("is_churned")
)

In [0]:
# Distribuicao de churn por tipo de assinatura
print("TAXA DE CHURN POR TIPO DE ASSINATURA:")
display(
    df_b.groupBy("subscription_type")
    .agg(
        count("*").alias("total"),
        spark_sum("is_churned").alias("churned"),
        spark_round(avg("is_churned") * 100, 2).alias("taxa_churn_pct")
    )
    .orderBy("taxa_churn_pct", ascending=False)
)

---
## PARTE 4 - Camada SILVER (Dados Limpos + Enriquecidos)

Na Silver fazemos o **Data Wrangling**: garantia de tipos, validacoes
de regras de negocio e criacao de features que transformam dados brutos
em sinais interpretaveis de risco de churn.

### Analogia: o prontuario do medico

O dado bruto e como uma ficha de anamnese baguncada. Na Silver,
um enfermeiro organiza tudo em prontuario digital - correto, padronizado,
enriquecido com indicadores clinicos. O medico (analista/modelo) so
ve o prontuario organizado, nunca a ficha original.

In [0]:
# ============================================================
# PASSO 1 de 6: Garantir tipos corretos
# ============================================================
# Mesmo com inferSchema=True, e boa pratica garantir os tipos
# explicitamente para evitar surpresas em novos arquivos.

df_silver = spark.table("workspace.default.bronze_usuarios_spotify")

df_silver = df_silver \
    .withColumn("user_id",               col("user_id").cast(IntegerType())) \
    .withColumn("age",                   col("age").cast(IntegerType())) \
    .withColumn("listening_time",        col("listening_time").cast(IntegerType())) \
    .withColumn("songs_played_per_day",  col("songs_played_per_day").cast(IntegerType())) \
    .withColumn("skip_rate",             col("skip_rate").cast(DoubleType())) \
    .withColumn("ads_listened_per_week", col("ads_listened_per_week").cast(IntegerType())) \
    .withColumn("offline_listening",     col("offline_listening").cast(IntegerType())) \
    .withColumn("is_churned",            col("is_churned").cast(IntegerType()))

print("Passo 1/6 concluido: Tipos de dados normalizados")
df_silver.printSchema()

In [0]:
# ============================================================
# PASSO 2 de 6: Flags de qualidade por registro
# ============================================================
# Criamos flags booleanas indicando se cada registro passa nas
# regras de negocio. Nao removemos registros - apenas sinalizamos.
# Isso preserva o volume da Bronze e mantem rastreabilidade.
#
# Regras:
#   - Idade entre 10 e 100 anos
#   - skip_rate entre 0.0 e 1.0 (e uma proporcao)
#   - listening_time nao-negativo
#   - songs_played_per_day nao-negativo

df_silver = df_silver \
    .withColumn("qa_idade_valida",
        when((col("age") >= 10) & (col("age") <= 100), True).otherwise(False)) \
    .withColumn("qa_skip_valido",
        when((col("skip_rate") >= 0.0) & (col("skip_rate") <= 1.0), True).otherwise(False)) \
    .withColumn("qa_listen_valido",
        when(col("listening_time") >= 0, True).otherwise(False)) \
    .withColumn("qa_songs_valido",
        when(col("songs_played_per_day") >= 0, True).otherwise(False)) \
    .withColumn("qa_registro_ok",
        when(
            (col("qa_idade_valida")  == True) &
            (col("qa_skip_valido")   == True) &
            (col("qa_listen_valido") == True) &
            (col("qa_songs_valido")  == True) &
            col("is_churned").isNotNull() &
            col("user_id").isNotNull(),
            True
        ).otherwise(False)
    )

qa_ok   = df_silver.filter(col("qa_registro_ok") == True).count()
qa_fail = df_silver.filter(col("qa_registro_ok") == False).count()
print(f"Passo 2/6 concluido: Flags de qualidade criadas")
print(f"   Registros validos:   {qa_ok:,}")
print(f"   Registros invalidos: {qa_fail:,}")

In [0]:
# ============================================================
# PASSO 3 de 6: Calcular thresholds estatisticos
# ============================================================
# Usamos percentis calculados dinamicamente - assim o pipeline
# se adapta automaticamente a novos volumes de dados.
# Calculamos apenas sobre registros validos (qa_registro_ok).

stats = df_silver.filter(col("qa_registro_ok") == True).agg(
    percentile_approx("listening_time",        0.25).alias("p25_listen"),
    percentile_approx("listening_time",        0.75).alias("p75_listen"),
    percentile_approx("skip_rate",             0.75).alias("p75_skip"),
    percentile_approx("songs_played_per_day",  0.25).alias("p25_songs"),
    percentile_approx("ads_listened_per_week", 0.75).alias("p75_ads"),
    avg("listening_time").alias("media_listen"),
    avg("skip_rate").alias("media_skip"),
).collect()[0]

p25_listen = stats["p25_listen"]
p75_listen = stats["p75_listen"]
p75_skip   = stats["p75_skip"]
p25_songs  = stats["p25_songs"]
p75_ads    = stats["p75_ads"]

print("Passo 3/6 concluido: Thresholds calculados")
print()
print("   Limiares de comportamento (base: dados validos):")
print(f"     Escuta baixa (p25):          {p25_listen:>8.0f} min totais")
print(f"     Escuta alta (p75):           {p75_listen:>8.0f} min totais")
print(f"     Skip alto (p75):             {p75_skip:>8.3f} (proporcao 0-1)")
print(f"     Musicas baixo (p25):         {p25_songs:>8.0f} musicas/dia")
print(f"     Anuncios alto (p75):         {p75_ads:>8.0f} anuncios/sem")

In [0]:
# ============================================================
# PASSO 4 de 6: Criar 6 flags de risco de churn
# ============================================================
# Cada flag representa um sinal observavel de desengajamento.
# A logica reflete o que estudos de retencao mostram:
# usuarios que ouvem pouco, pulam muito e toleram muitos anuncios
# tem maior propensao a cancelar.

# Flag 1: Escuta muito baixa - nao consome o produto
df_silver = df_silver.withColumn("flag_escuta_baixa",
    when(col("listening_time") < p25_listen, 1).otherwise(0))

# Flag 2: Alta taxa de pulos - insatisfeito com as recomendacoes
df_silver = df_silver.withColumn("flag_skip_alto",
    when(col("skip_rate") > p75_skip, 1).otherwise(0))

# Flag 3: Poucas musicas por dia - engajamento minimo
df_silver = df_silver.withColumn("flag_pouco_consumo",
    when(col("songs_played_per_day") < p25_songs, 1).otherwise(0))

# Flag 4: Muitos anuncios ouvidos - usuario Free sem converter
df_silver = df_silver.withColumn("flag_ads_alto",
    when(
        (col("ads_listened_per_week") > p75_ads) &
        (col("subscription_type") == "Free"),
        1
    ).otherwise(0))

# Flag 5: Nao usa modo offline - baixo comprometimento com a plataforma
df_silver = df_silver.withColumn("flag_sem_offline",
    when(col("offline_listening") == 0, 1).otherwise(0))

# Flag 6: Skip alto + escuta baixa combinados (sinal duplo de abandono)
df_silver = df_silver.withColumn("flag_abandono_duplo",
    when(
        (col("skip_rate") > p75_skip) &
        (col("listening_time") < p25_listen),
        1
    ).otherwise(0))

print("Passo 4/6 concluido: 6 flags de risco criadas")
print()
total_silver = df_silver.count()
for flag in ["flag_escuta_baixa", "flag_skip_alto", "flag_pouco_consumo",
             "flag_ads_alto", "flag_sem_offline", "flag_abandono_duplo"]:
    qtd = df_silver.filter(col(flag) == 1).count()
    pct = qtd / total_silver * 100
    print(f"     {flag:35s}  ->  {qtd:>8,} usuarios ({pct:.1f}%)")

In [0]:
# ============================================================
# PASSO 5 de 6: Score de risco de churn consolidado (0 a 6)
# ============================================================
# Somamos as flags - quanto maior o score, mais sinais de churn.
# O nivel e usado para segmentar a acao do time de CRM.

df_silver = df_silver.withColumn("score_churn",
    col("flag_escuta_baixa") +
    col("flag_skip_alto") +
    col("flag_pouco_consumo") +
    col("flag_ads_alto") +
    col("flag_sem_offline") +
    col("flag_abandono_duplo")
)

df_silver = df_silver.withColumn("nivel_risco_churn",
    when(col("score_churn") >= 5, "CRITICO")
    .when(col("score_churn") >= 4, "ALTO")
    .when(col("score_churn") >= 2, "MEDIO")
    .when(col("score_churn") >= 1, "BAIXO")
    .otherwise("SAUDAVEL")
)

print("Passo 5/6 concluido: Score de churn calculado")
print()
df_silver.groupBy("nivel_risco_churn") \
    .agg(count("*").alias("quantidade")) \
    .orderBy("quantidade", ascending=False) \
    .display()

In [0]:
# ============================================================
# PASSO 6 de 6: Salvar camada Silver
# ============================================================

df_silver = df_silver.withColumn("_silver_timestamp", current_timestamp())

df_silver.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_usuarios_spotify")

total_s = spark.table("workspace.default.silver_usuarios_spotify").count()
print(f"Passo 6/6 concluido: Camada SILVER salva!")
print(f"   Registros:     {total_s:,}")
print(f"   Tabela:        workspace.default.silver_usuarios_spotify")

In [0]:
# Amostra da Silver com flags e score
display(spark.table("workspace.default.silver_usuarios_spotify").limit(20))

 
---
## PARTE 5 - Analise de Eventos de E-commerce (Camada Silver Complementar)

O arquivo `events.csv` contem dados de comportamento de e-commerce
(visualizacoes, carrinho, compras) com schema diferente do dataset de usuarios.
Como os `user_id` sao de origens distintas, **nao fazemos join** com a Silver
de usuarios. Em vez disso, construimos uma **Silver de eventos independente**
que entrega metricas de funil de conversao por categoria e marca.

Essa e uma situacao comum em projetos reais: nem sempre todas as fontes
se conectam - e ter insights separados ja tem valor proprio para o negocio.

### Schema dos eventos (ecommerce behavior)

| Coluna | Descricao |
|--------|-----------|
| `event_time` | Timestamp do evento |
| `event_type` | Tipo: view / cart / purchase / remove_from_cart |
| `product_id` | ID do produto |
| `category_id` | ID da categoria |
| `category_code` | Caminho da categoria (ex: electronics.telephone) |
| `brand` | Marca do produto |
| `price` | Preco do produto |
| `user_id` | ID do usuario (escopo deste dataset) |
| `user_session` | ID da sessao do usuario |

In [0]:
df_ev = spark.table("workspace.default.bronze_eventos_spotify")
 
print(f"Total de eventos: {df_ev.count():,}")
print(f"Colunas: {df_ev.columns}")
print()
 
# Distribuicao dos tipos de evento
print("Distribuicao de event_type:")
display(
    df_ev.groupBy("event_type")
    .agg(
        count("*").alias("total"),
        spark_round(count("*") / df_ev.count() * 100, 2).alias("percentual")
    )
    .orderBy("total", ascending=False)
)

 
%md
### Limpeza dos eventos
Antes de agregar, sinalizamos registros com preco invalido
e categoria nula - ambos sao comuns em datasets de e-commerce.


In [0]:

# Flags de qualidade nos eventos
df_ev_clean = df_ev \
    .withColumn("qa_preco_valido",
        when((col("price") > 0) & col("price").isNotNull(), True).otherwise(False)) \
    .withColumn("qa_categoria_ok",
        when(col("category_code").isNotNull(), True).otherwise(False)) \
    .withColumn("qa_evento_ok",
        when(
            col("event_type").isin("view", "cart", "purchase", "remove_from_cart") &
            col("user_id").isNotNull() &
            col("product_id").isNotNull(),
            True
        ).otherwise(False)
    )
 
ev_ok   = df_ev_clean.filter(col("qa_evento_ok") == True).count()
ev_fail = df_ev_clean.filter(col("qa_evento_ok") == False).count()
total_ev = df_ev_clean.count()
 
print(f"Qualidade dos eventos:")
print(f"   Validos:   {ev_ok:,} ({ev_ok/total_ev*100:.1f}%)")
print(f"   Invalidos: {ev_fail:,} ({ev_fail/total_ev*100:.1f}%)")


### Construção da Silver de Eventos

Agregamos metricas de funil de conversao por usuario dentro do escopo deste dataset de e-commerce.

In [0]:
# Trabalhar apenas com eventos validos
df_ev_validos = df_ev_clean.filter(col("qa_evento_ok") == True)
 
# Agregar funil de conversao por usuario
df_silver_eventos = df_ev_validos.groupBy("user_id").agg(
    count("*").alias("total_eventos"),
    countDistinct("user_session").alias("total_sessoes"),
    spark_sum(when(col("event_type") == "view",     1).otherwise(0)).alias("total_views"),
    spark_sum(when(col("event_type") == "cart",     1).otherwise(0)).alias("total_carrinhos"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("total_compras"),
    spark_sum(when(col("event_type") == "remove_from_cart", 1).otherwise(0)).alias("total_remocoes"),
    countDistinct("product_id").alias("produtos_distintos"),
    countDistinct("brand").alias("marcas_distintas"),
    spark_round(avg(when(col("event_type") == "purchase", col("price"))), 2).alias("ticket_medio"),
    spark_round(spark_sum(when(col("event_type") == "purchase", col("price"))), 2).alias("gasto_total"),
) \
.withColumn("taxa_conversao_pct",
    spark_round(
        when(col("total_views") > 0,
            col("total_compras") / col("total_views") * 100
        ).otherwise(0.0),
    2)
) \
.withColumn("taxa_abandono_carrinho_pct",
    spark_round(
        when(col("total_carrinhos") > 0,
            col("total_remocoes") / col("total_carrinhos") * 100
        ).otherwise(0.0),
    2)
)
 
df_silver_eventos.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_eventos_ecommerce")
 
print(f"Silver Eventos criada!")
print(f"   Usuarios unicos: {df_silver_eventos.count():,}")
print(f"   Tabela: workspace.default.silver_eventos_ecommerce")
display(df_silver_eventos.limit(10))

### Analise por Categoria e Marca (Gold Complementar)

In [0]:
from pyspark.sql.functions import col, when, split, count, sum as spark_sum, avg, round as spark_round

gold_categorias = df_ev_validos \
    .filter(col("category_code").isNotNull()) \
    .withColumn(
        "categoria_nivel1",
        when(
            col("category_code").contains("."),
            split(col("category_code"), "\\.").getItem(0)
        ).otherwise(col("category_code"))
    ) \
    .groupBy("category_code", "categoria_nivel1") \
    .agg(
        count("*").alias("total_eventos"),
        spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
        spark_sum(when(col("event_type") == "cart", 1).otherwise(0)).alias("carrinhos"),
        spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("compras"),
        spark_round(avg(when(col("event_type") == "purchase", col("price"))), 2).alias("preco_medio"),
        spark_round(spark_sum(when(col("event_type") == "purchase", col("price"))), 2).alias("receita_total")
    ) \
    .withColumn(
        "conversao_view_compra_pct",
        spark_round(
            when(col("views") > 0, col("compras") / col("views") * 100).otherwise(0), 2
        )
    ) \
    .orderBy(col("receita_total"), ascending=False)

gold_categorias.write.mode("overwrite").saveAsTable("workspace.default.gold_eventos_categorias")

print("Gold Categorias criada!")
display(gold_categorias.limit(15))

# Gold: top marcas por volume de compras e receita

In [0]:
gold_marcas = df_ev_validos \
    .filter(
        col("brand").isNotNull() &
        (col("event_type") == "purchase")
    ) \
    .groupBy("brand").agg(
        count("*").alias("total_compras"),
        countDistinct("user_id").alias("compradores_unicos"),
        spark_round(avg("price"), 2).alias("preco_medio"),
        spark_round(spark_sum("price"), 2).alias("receita_total"),
    ) \
    .orderBy("receita_total", ascending=False)
 
gold_marcas.write.mode("overwrite").saveAsTable("workspace.default.gold_eventos_marcas")
 
print("Gold Marcas criada!")
display(gold_marcas.limit(15))

---
## PARTE 6 - Camada GOLD (Metricas de Negocio)

A Gold e a camada que o **negocio consome**. Aqui so existem metricas
prontas para decisao - nada de dado bruto ou intermediario.
O time de CRM le a Gold diretamente, sem precisar entender o pipeline.

In [0]:
df_s = spark.table("workspace.default.silver_usuarios_spotify")

# ============================================================
# GOLD 1: Eficacia do score - nosso modelo detecta churn?
# ============================================================

gold_eficacia = df_s.groupBy("nivel_risco_churn").agg(
    count("*").alias("total_usuarios"),
    spark_sum("is_churned").alias("churned_reais"),
    spark_round(avg("is_churned") * 100, 2).alias("taxa_churn_pct"),
    spark_round(avg("listening_time"), 0).alias("escuta_media_min"),
    spark_round(avg("skip_rate"), 3).alias("skip_medio"),
    spark_round(avg("songs_played_per_day"), 1).alias("musicas_media_dia"),
).orderBy("taxa_churn_pct", ascending=False)

gold_eficacia.write.mode("overwrite").saveAsTable("workspace.default.gold_eficacia_churn")

print("Gold Table 1: Analise de eficacia do score de churn")
display(gold_eficacia)

In [0]:
# ============================================================
# GOLD 2: Dashboard operacional para o time de retencao
# ============================================================

total       = df_s.count()
total_churn = df_s.filter(col("is_churned") == 1).count()
total_crit  = df_s.filter(col("nivel_risco_churn") == "CRITICO").count()
total_alto  = df_s.filter(col("nivel_risco_churn") == "ALTO").count()

churn_capturado = df_s.filter(
    (col("is_churned") == 1) & (col("score_churn") >= 4)
).count()
taxa_deteccao = (churn_capturado / total_churn * 100) if total_churn > 0 else 0

falsos_positivos = df_s.filter(
    (col("is_churned") == 0) & (col("score_churn") >= 4)
).count()

# Criar DataFrame para dashboard
dashboard_data = [{
    "total_usuarios":              total,
    "usuarios_churn":              total_churn,
    "taxa_churn_pct":              round(total_churn / total * 100, 2),
    "usuarios_criticos":           total_crit,
    "usuarios_alto_risco":         total_alto,
    "churn_capturado":             churn_capturado,
    "taxa_deteccao_pct":           round(taxa_deteccao, 2),
    "falsos_positivos":            falsos_positivos,
}]

df_dashboard = spark.createDataFrame(dashboard_data)
display(df_dashboard)

# Guardar metricas para os quality checks e log
metricas = {
    "total_usuarios":    total,
    "total_churn":       total_churn,
    "taxa_churn_pct":    round(total_churn / total * 100, 2),
    "usuarios_criticos": total_crit,
    "churn_capturado":   churn_capturado,
    "taxa_deteccao_pct": round(taxa_deteccao, 2),
    "falsos_positivos":  falsos_positivos,
}

In [0]:
# ============================================================
# GOLD 3: Lista de usuarios prioritarios para acao do CRM
# ============================================================

gold_lista_churn = df_s \
    .filter(col("score_churn") >= 4) \
    .select(
        "user_id", "subscription_type", "country", "age",
        "listening_time", "skip_rate", "songs_played_per_day",
        "ads_listened_per_week", "offline_listening",
        "score_churn", "nivel_risco_churn", "is_churned"
    ) \
    .orderBy("score_churn", ascending=False)

gold_lista_churn.write.mode("overwrite").saveAsTable("workspace.default.gold_lista_churn")

print(f"Usuarios para acao imediata: {gold_lista_churn.count():,}")
display(gold_lista_churn.limit(20))

In [0]:
# ============================================================
# GOLD 4: Segmentacao por tipo de assinatura + perfil de risco
# ============================================================

gold_segmentos = df_s.groupBy("subscription_type", "nivel_risco_churn").agg(
    count("*").alias("total"),
    spark_sum("is_churned").alias("churned"),
    spark_round(avg("is_churned") * 100, 2).alias("taxa_churn_pct"),
    spark_round(avg("listening_time"), 0).alias("escuta_media"),
    spark_round(avg("skip_rate"), 3).alias("skip_medio"),
    spark_round(avg("ads_listened_per_week"), 1).alias("ads_media"),
).orderBy("subscription_type", "taxa_churn_pct", ascending=[True, False])

gold_segmentos.write.mode("overwrite").saveAsTable("workspace.default.gold_segmentos_churn")

print("Gold Table 4: Segmentacao por assinatura + nivel de risco")
display(gold_segmentos)

---
## PARTE 7 - Quality Gate (Validacoes Automaticas)

Verificacoes que confirmam que os dados fazem sentido de ponta a ponta.
Se algo falhar, o pipeline registra o alerta em vez de propagar
dado errado silenciosamente para o time de negocio.

In [0]:
def executar_quality_checks():
    """
    8 verificacoes de qualidade no pipeline de churn.
    Retorna lista de resultados e lista de alertas criticos.
    """
    df_b    = spark.table("workspace.default.bronze_usuarios_spotify")
    df_s    = spark.table("workspace.default.silver_usuarios_spotify")
    total_b = df_b.count()
    total_s = df_s.count()
    resultados = []
    alertas    = []

    # CHECK 1: Consistencia Bronze -> Silver (sem perda de registros)
    diff = abs(total_b - total_s)
    ok = diff == 0
    resultados.append({
        "check": "Consistencia Bronze == Silver",
        "status": "PASS" if ok else "FAIL",
        "detalhe": f"Bronze: {total_b:,} | Silver: {total_s:,} | Diff: {diff}"
    })
    if not ok:
        alertas.append(f"FAIL - Perda de {diff} registros entre Bronze e Silver!")

    # CHECK 2: Campo is_churned sem nulos
    nulos_churn = df_s.filter(col("is_churned").isNull()).count()
    ok = nulos_churn == 0
    resultados.append({
        "check": "Campo is_churned sem nulos",
        "status": "PASS" if ok else "FAIL",
        "detalhe": f"{nulos_churn} nulos"
    })
    if not ok:
        alertas.append(f"FAIL - Campo is_churned tem {nulos_churn} nulos!")

    # CHECK 3: Taxa de churn no range esperado para streaming (3-40%)
    taxa = df_s.filter(col("is_churned") == 1).count() / total_s * 100
    ok = 3.0 <= taxa <= 40.0
    resultados.append({
        "check": "Taxa de churn entre 3% e 40%",
        "status": "PASS" if ok else "WARN",
        "detalhe": f"{taxa:.2f}%"
    })
    if not ok:
        alertas.append(f"WARN - Taxa de churn fora do range esperado: {taxa:.2f}%")

    # CHECK 4: user_id unico (sem duplicatas)
    total_ids = df_s.count()
    distintos = df_s.select("user_id").distinct().count()
    ok = total_ids == distintos
    resultados.append({
        "check": "user_id unico (sem duplicatas)",
        "status": "PASS" if ok else "FAIL",
        "detalhe": f"Total: {total_ids:,} | Distintos: {distintos:,}"
    })
    if not ok:
        alertas.append(f"FAIL - {total_ids - distintos:,} user_ids duplicados!")

    # CHECK 5: Score de churn calculado para todos os registros
    sem_score = df_s.filter(col("score_churn").isNull()).count()
    ok = sem_score == 0
    resultados.append({
        "check": "Score de churn calculado para todos",
        "status": "PASS" if ok else "FAIL",
        "detalhe": f"{sem_score} registros sem score"
    })

    # CHECK 6: Quality score geral > 95%
    qa_ok_cnt = df_s.filter(col("qa_registro_ok") == True).count()
    pct_ok    = qa_ok_cnt / total_s * 100
    ok = pct_ok >= 95
    resultados.append({
        "check": "Quality score geral > 95%",
        "status": "PASS" if ok else "WARN",
        "detalhe": f"{pct_ok:.2f}% aprovados"
    })
    if not ok:
        alertas.append(f"WARN - Apenas {pct_ok:.2f}% passaram no QA (esperado >95%)")

    # CHECK 7: Tempo medio de escuta deve ser positivo
    media_listen = df_s.agg(avg("listening_time")).collect()[0][0]
    ok = media_listen > 0 if media_listen is not None else False
    resultados.append({
        "check": "Tempo medio de escuta > 0",
        "status": "PASS" if ok else "FAIL",
        "detalhe": f"Media: {media_listen:.1f} min" if media_listen else "Null"
    })

    # CHECK 8: Percentual de CRITICOS nao deve passar de 15%
    pct_crit = df_s.filter(col("nivel_risco_churn") == "CRITICO").count() / total_s * 100
    ok = pct_crit <= 15.0
    resultados.append({
        "check": "CRITICOS <= 15% da base",
        "status": "PASS" if ok else "WARN",
        "detalhe": f"{pct_crit:.2f}% sao CRITICOS"
    })
    if not ok:
        alertas.append(f"WARN - {pct_crit:.2f}% dos usuarios sao CRITICOS (>15% preocupante)")

    return resultados, alertas


resultados, alertas = executar_quality_checks()

print("=" * 78)
print("  RELATORIO DE QUALIDADE - PIPELINE CHURN SPOTIFY")
print("=" * 78)
for r in resultados:
    ic = "PASS" if r["status"] == "PASS" else "WARN" if r["status"] == "WARN" else "FAIL"
    print(f"  [{ic}] {r['check']:45s} | {r['detalhe']}")
print("=" * 78)

tp = sum(1 for r in resultados if r["status"] == "PASS")
tw = sum(1 for r in resultados if r["status"] == "WARN")
tf = sum(1 for r in resultados if r["status"] == "FAIL")
print(f"\n  Resultado: {tp} PASS  |  {tw} WARN  |  {tf} FAIL")

if alertas:
    print(f"\n  ALERTAS ({len(alertas)}):")
    for a in alertas:
        print(f"     {a}")
else:
    print(f"\n  Nenhum alerta - pipeline saudavel!")

---
## PARTE 8 - Registro de Alertas (Log de Execucao)

Gravamos o resultado de cada execucao numa tabela Delta.
Isso cria um historico auditavel - qualquer gestor pode verificar
quando o pipeline rodou, quantos checks passaram e se houve anomalia.

In [0]:
alerta_data = [{
    "pipeline_nome":      "Pipeline_Churn_Spotify",
    "timestamp_execucao": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "total_checks":       len(resultados),
    "checks_passed":      sum(1 for r in resultados if r["status"] == "PASS"),
    "checks_failed":      sum(1 for r in resultados if r["status"] == "FAIL"),
    "checks_warned":      sum(1 for r in resultados if r["status"] == "WARN"),
    "total_usuarios":     metricas["total_usuarios"],
    "total_churn":        metricas["total_churn"],
    "taxa_churn_pct":     metricas["taxa_churn_pct"],
    "taxa_deteccao_pct":  metricas["taxa_deteccao_pct"],
    "alertas_detalhe":    json_lib.dumps(alertas, ensure_ascii=False),
    "resultado":          "OK" if not alertas else "ATENCAO",
}]

df_log = spark.createDataFrame(alerta_data)
# mode("append") acumula historico de execucoes
df_log.write.mode("append").saveAsTable("workspace.default.log_pipeline_churn")

print("Log de execucao registrado!")
print(f"   Tabela: workspace.default.log_pipeline_churn")
display(df_log)

---
## PARTE 9 - Consultas SQL de Negocio

In [0]:
%sql
-- 1. Eficacia do score: quanto maior o nivel, mais churn?
SELECT
    nivel_risco_churn,
    total_usuarios,
    churned_reais,
    taxa_churn_pct,
    escuta_media_min,
    skip_medio,
    musicas_media_dia
FROM workspace.default.gold_eficacia_churn
ORDER BY taxa_churn_pct DESC

In [0]:
%sql
-- 2. Qual combinacao de fatores tem MAIOR taxa de churn?
SELECT
    CASE WHEN subscription_type = 'Premium' THEN 'Premium'
         WHEN subscription_type = 'Family'  THEN 'Family'
         ELSE 'Free' END                         AS plano,
    CASE WHEN flag_escuta_baixa = 1 THEN 'Pouco ouve'
         ELSE 'Ouve bastante' END                 AS escuta,
    CASE WHEN flag_skip_alto = 1 THEN 'Skip alto'
         ELSE 'Skip ok' END                       AS pulos,
    COUNT(*)                                      AS total,
    SUM(is_churned)                               AS churned,
    ROUND(AVG(is_churned) * 100, 2)               AS taxa_churn_pct
FROM workspace.default.silver_usuarios_spotify
GROUP BY subscription_type, flag_escuta_baixa, flag_skip_alto
ORDER BY taxa_churn_pct DESC
LIMIT 10

In [0]:
%sql
-- 3. Top 20 usuarios mais criticos para o CRM agir agora
SELECT
    user_id,
    subscription_type,
    country,
    age,
    score_churn,
    nivel_risco_churn,
    listening_time                  AS escuta_total_min,
    ROUND(skip_rate * 100, 1)       AS skip_pct,
    songs_played_per_day            AS musicas_dia,
    ads_listened_per_week           AS anuncios_sem,
    is_churned
FROM workspace.default.gold_lista_churn
ORDER BY score_churn DESC
LIMIT 20

In [0]:
%sql
-- 4. Perfil tipico do usuario que da churn vs que permanece
SELECT
    CASE WHEN is_churned = 1 THEN 'Churned' ELSE 'Ativo' END AS status_usuario,
    COUNT(*)                                AS total,
    ROUND(AVG(age), 0)                      AS idade_media,
    ROUND(AVG(listening_time), 0)           AS escuta_total_media,
    ROUND(AVG(skip_rate) * 100, 1)          AS skip_pct_medio,
    ROUND(AVG(songs_played_per_day), 1)     AS musicas_dia_media,
    ROUND(AVG(ads_listened_per_week), 1)    AS anuncios_sem_medio,
    SUM(offline_listening)                  AS usam_offline,
    ROUND(AVG(score_churn), 2)              AS score_medio
FROM workspace.default.silver_usuarios_spotify
GROUP BY is_churned
ORDER BY is_churned

In [0]:
%sql
-- 5. Churn por pais - onde a retencao e pior?
SELECT
    country,
    COUNT(*) AS total_usuarios,
    SUM(is_churned) AS churned,
    ROUND(AVG(is_churned) * 100, 2) AS taxa_churn_pct
FROM workspace.default.silver_usuarios_spotify
GROUP BY country
HAVING COUNT(*) >= 50
ORDER BY taxa_churn_pct DESC
LIMIT 15

In [0]:
%sql
-- 6. Historico de execucoes do pipeline (log de alertas)
SELECT * FROM workspace.default.log_pipeline_churn
ORDER BY timestamp_execucao DESC

---
## PARTE 10 - Delta Lake: Time Travel

O Delta Lake guarda versoes dos dados a cada escrita.
Voce pode "viajar no tempo" e acessar qualquer versao anterior.
No contexto de churn, isso permite comparar a base de risco
de hoje com a de semanas atras sem nenhum dado adicional.

In [0]:
# Historico de versoes da Silver
display(spark.sql("DESCRIBE HISTORY workspace.default.silver_usuarios_spotify"))

In [0]:
# Ler versao 0 da Silver (antes do enriquecimento com eventos)
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("workspace.default.silver_usuarios_spotify")

print(f"Versao 0 da Silver: {df_v0.count():,} registros")
print(f"Colunas: {df_v0.columns}")

---
## PARTE 11 - Governanca e Sustentabilidade

### Politica de expurgo e retencao

| Camada | Retencao | Justificativa |
|--------|----------|---------------|
| Bronze | 12 meses | Reprocessamento e auditoria; apos 12m arquivar em cold storage |
| Silver | 6 meses  | Features calculadas; historico recente e suficiente para CRM |
| Gold   | 24 meses | Metricas para comparacao historica e relatorios executivos |
| Log    | 24 meses | Auditoria de pipeline obrigatoria por compliance |

### Frequencia de atualizacao
- **Bronze:** Diaria - ingestao automatica via Databricks Jobs
- **Silver / Gold:** Diaria - job encadeado apos conclusao da Bronze
- **Log:** A cada execucao do pipeline

### Dados sensiveis
- `user_id`: identificador unico - em producao, aplicar hash SHA-256
  antes de expor em ambientes nao-produtivos
- `country`: dado geografico - anonimizar se necessario por LGPD/GDPR
- Controle de acesso via Unity Catalog: time de CRM acessa apenas Gold

In [0]:
# Simulacao: identificar usuarios sem nenhuma escuta (candidatos a expurgo)

candidatos_expurgo = spark.table("workspace.default.silver_usuarios_spotify") \
    .filter(col("listening_time") == 0) \
    .count()

total_s     = spark.table("workspace.default.silver_usuarios_spotify").count()
pct_expurgo = (candidatos_expurgo / total_s * 100) if total_s > 0 else 0

print("=" * 60)
print("  SIMULACAO DE POLITICA DE EXPURGO")
print("=" * 60)
print(f"  Criterio:     listening_time = 0 (sem uso registrado)")
print(f"  Total base:   {total_s:,}")
print(f"  Candidatos:   {candidatos_expurgo:,} ({pct_expurgo:.1f}%)")
print(f"  Remanescente: {total_s - candidatos_expurgo:,} ({100 - pct_expurgo:.1f}%)")
print("=" * 60)
print("  Em producao, esses registros seriam arquivados")
print("  em storage de baixo custo (ex: S3 Glacier).")

---
## PARTE 12 - Limpeza (Opcional)

Descomente para remover todas as tabelas criadas neste pipeline.

In [0]:
# Descomente para limpar:

# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_usuarios_spotify")
# spark.sql("DROP TABLE IF EXISTS workspace.default.bronze_eventos_spotify")
# spark.sql("DROP TABLE IF EXISTS workspace.default.silver_usuarios_spotify")
# spark.sql("DROP TABLE IF EXISTS workspace.default.gold_eficacia_churn")
# spark.sql("DROP TABLE IF EXISTS workspace.default.gold_lista_churn")
# spark.sql("DROP TABLE IF EXISTS workspace.default.gold_segmentos_churn")
# spark.sql("DROP TABLE IF EXISTS workspace.default.log_pipeline_churn")
# print("Tabelas removidas!")

---
## Resumo do Pipeline

| Etapa | Tabela UC | Descricao |
|-------|-----------|-----------|
| Bronze | `bronze_usuarios_spotify` | Dados brutos de usuarios + metadados de ingestao |
| Bronze | `bronze_eventos_spotify` | Eventos brutos + metadados de ingestao |
| Silver | `silver_usuarios_spotify` | Limpo + 6 flags de churn + score + eventos agregados |
| Gold | `gold_eficacia_churn` | Eficacia do score por nivel de risco |
| Gold | `gold_lista_churn` | Usuarios prioritarios para campanha de retencao |
| Gold | `gold_segmentos_churn` | Segmentacao por plano x nivel de risco |
| Log | `log_pipeline_churn` | Historico auditavel de execucoes e alertas |

### Consumidores das camadas Gold
- **Time de CRM:** `gold_lista_churn` - campanhas de retencao personalizadas
- **Time de Produto:** `gold_segmentos_churn` - decisoes de roadmap e pricing
- **Engenharia de Dados:** `log_pipeline_churn` - monitoramento de saude do pipeline

---
**Disciplina:** Data Integration e Pipelines - MBA Data Engineering


## Resumo do Pipeline

### Fluxo de Churn (spotify_churn_dataset.csv)

| Etapa | Tabela UC | Descricao |
|-------|-----------|-----------|
| Bronze | `bronze_usuarios_spotify` | Dados brutos de usuarios + metadados |
| Silver | `silver_usuarios_spotify` | Limpo + 6 flags de churn + score de risco |
| Gold | `gold_eficacia_churn` | Eficacia do score por nivel de risco |
| Gold | `gold_lista_churn` | Usuarios prioritarios para campanha de retencao |
| Gold | `gold_segmentos_churn` | Segmentacao por plano x nivel de risco |
| Log | `log_pipeline_churn` | Historico auditavel de execucoes e alertas |

### Fluxo de Eventos E-commerce (events.csv)

| Etapa | Tabela UC | Descricao |
|-------|-----------|-----------|
| Bronze | `bronze_eventos_spotify` | Eventos brutos de e-commerce + metadados |
| Silver | `silver_eventos_ecommerce` | Funil de conversao agregado por usuario |
| Gold | `gold_eventos_categorias` | Conversao e receita por categoria de produto |
| Gold | `gold_eventos_marcas` | Top marcas por volume de compras e receita |

### Consumidores das camadas Gold
- **Time de CRM:** `gold_lista_churn` - campanhas de retencao personalizadas
- **Time de Produto:** `gold_segmentos_churn` + `gold_eventos_categorias` - roadmap e pricing
- **Time Comercial:** `gold_eventos_marcas` - negociacao com fornecedores
- **Engenharia de Dados:** `log_pipeline_churn` - monitoramento de saude do pipeline

---
**Disciplina:** Data Integration e Pipelines - MBA Data Engineering